# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [192]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [193]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 467.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [194]:
query_embeddings.shape

(5346, 384)

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [195]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [196]:
import redis

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

r.flushall()


True

# 1. Funciones

In [197]:
def cargar_cartas(file_path):

    df = pd.read_csv(file_path)
    df.drop_duplicates(inplace=True)
    print(f"Iniciando la carga de {len(df)} registros...")

    for _, row in df.iterrows():
        clave = f"card:{row['code']}"
        r.hset(clave, mapping = row.to_dict())

    # ver el numero de cartas importadas
    keys = r.keys("card:*")
    print(f"Se han importado {len(keys)} cartas en Redis.")


cargar_cartas("cards_final_with_xp.csv")

Iniciando la carga de 5324 registros...
Se han importado 5324 cartas en Redis.


In [198]:
def card_exists(code):
    return r.exists(f"card:{code}") == 1

card_exists("60401")

True

In [199]:
def ver_carta_code(code, verbose=False):
    if card_exists(code):
        card_data = r.hgetall(f"card:{code}")
        

        if verbose:
            print(f"Datos de la carta {code}:")
            for key, value in card_data.items():
                print(f"{key}: {value}")
            display_card(card_data["image_url"])

        return card_data
    
    print(f"No se encontró la carta con código {code}.")
    
    
        
        

ver_carta_code("60402", verbose=True)


Datos de la carta 60402:
pack_code: jac
image_url: https://arkhamdb.com/bundles/cards/60402.jpg
name: Arbiter of Fates
illustrator: Pavel Kolomeyets
code: 60402
type_code: asset
traits: Talent
faction_code: mystic
xp: 0
text: Jacqueline Fine deck only. [reaction] When you use Jacqueline Fine's [reaction] ability, exhaust Arbiter of Fates: This use of her ability does not count towards its limit.


{'pack_code': 'jac',
 'image_url': 'https://arkhamdb.com/bundles/cards/60402.jpg',
 'name': 'Arbiter of Fates',
 'illustrator': 'Pavel Kolomeyets',
 'code': '60402',
 'type_code': 'asset',
 'traits': 'Talent',
 'faction_code': 'mystic',
 'xp': '0',
 'text': "Jacqueline Fine deck only. [reaction] When you use Jacqueline Fine's [reaction] ability, exhaust Arbiter of Fates: This use of her ability does not count towards its limit."}

In [200]:
def meter_carta(code, name, text, type_code, traits, pack_code, faction_code, xp, illustrator, image_url):
    card_data = {
        "code": code,
        "name": name,
        "text": text,
        "type_code": type_code,
        "traits": traits,
        "pack_code": pack_code,
        "illustrator": illustrator,
        "image_url": image_url,
        "xp": xp,
        "faction_code": faction_code
    }
    r.hset(f"card:{code}", mapping=card_data)

meter_carta("99999", "Carta de Prueba", "Este es un texto de prueba.", "type_test", "trait1, trait2", "pack_test", "faction_test|faction_test2", 5, "Test Illustrator", "http://example.com/image.jpg")
ver_carta_code("99999")

{'code': '99999',
 'name': 'Carta de Prueba',
 'text': 'Este es un texto de prueba.',
 'type_code': 'type_test',
 'traits': 'trait1, trait2',
 'pack_code': 'pack_test',
 'illustrator': 'Test Illustrator',
 'image_url': 'http://example.com/image.jpg',
 'xp': '5',
 'faction_code': 'faction_test|faction_test2'}

In [201]:
def eliminar_carta(code):
    if card_exists(code):
        r.delete(f"card:{code}")
        print(f"Carta con código {code} eliminada.")
    else:
        print(f"No se encontró la carta con código {code} para eliminar.")

eliminar_carta("99999")

ver_carta_code("99999")

Carta con código 99999 eliminada.
No se encontró la carta con código 99999.


# Indice

In [202]:
from redis.commands.search.field import TextField, TagField, NumericField
from redis.commands.search.index_definition import IndexDefinition, IndexType
import time

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 1. Definimos el esquema (qué columnas queremos indexar)

schema = (
            TagField("code", sortable=True),           # Tag ya que hay algunas que 
            TextField("name", sortable=True),  # TEXT para búsquedas de palabras (el nombre pesa más 
            TextField("text"),              # TEXT para el cuerpo de la carta

            TagField("traits", separator="|"), # TAG indicando que los rasgos se separan por '|'

            TagField("faction_code", separator="|"),       # TAG para la facción a la que pertenece la carta
            NumericField("xp", sortable=True),          # NUMERIC para filtros de rango (ej: xp > 2)
            TextField("image_url")        # TEXT para la URL de la imagen de la carta  
        )

try:
    r.ft("cartas:").create_index(
        schema,
        
        definition=IndexDefinition(
            prefix=["card:"],       # SOLO indexará llaves que empiecen por "card:"
            index_type=IndexType.HASH
        )
    )
    time.sleep(1)  # Pequeña pausa para asegurarnos de que el índice se ha creado antes de hacer consultas

    print("✅ Índice creado con éxito")
except Exception as e:
    print(f"❌ Error o el índice ya existe: {e}")

✅ Índice creado con éxito


In [203]:
ver_carta_code("60402")

{'pack_code': 'jac',
 'faction_code': 'mystic',
 'name': 'Arbiter of Fates',
 'illustrator': 'Pavel Kolomeyets',
 'code': '60402',
 'xp': '0',
 'text': "Jacqueline Fine deck only. [reaction] When you use Jacqueline Fine's [reaction] ability, exhaust Arbiter of Fates: This use of her ability does not count towards its limit.",
 'image_url': 'https://arkhamdb.com/bundles/cards/60402.jpg',
 'type_code': 'asset',
 'traits': 'Talent'}

In [ ]:
from redis.commands.search.aggregation import AggregateRequest
from redis.commands.search.aggregation import Asc, Desc


def buscar_cartas_faccion(*factions, page_size=5, page_number=0):

    offset = page_number * page_size

    request = AggregateRequest("*").load("@code", "@faction_code")

    if factions:
        
        condiciones = [f'contains(@faction_code, "{f}")' for f in factions]

        request.filter("&&".join(condiciones))
        
    else:
        request.filter('contains(@faction_code, "|")')

    request.sort_by("@code", asc=True)

    request.limit(offset, page_size)

    response = r.ft("cartas:").aggregate(request)

    solucion = response.rows
    return solucion

# Ejecución

page_number = 0
page_size = 5

resultado = buscar_cartas_faccion(page_number=page_number, page_size=page_size)
resultado

[['code', '08083', 'faction_code', 'guardian|seeker'],
 ['code', '08084', 'faction_code', 'guardian|seeker'],
 ['code', '08085', 'faction_code', 'guardian|seeker'],
 ['code', '08086', 'faction_code', 'guardian|seeker'],
 ['code', '08087', 'faction_code', 'guardian|rogue']]